Selected features of dataset cicids2017_cleaned.csv 
1.  Bwd Packet Length Std 
2.  Subflow Fwd Bytes 
3.  Flow Duration
4.  Total Length of Fwd Packets
5.  Init_Win_bytes_forward
6.  Flow IAT Std          
7.  Active Mean           
8.  Bwd Packets/s         
9.  Fwd Packet Length Mean
10. Bwd Packet Length Min 
11. Attack Type (label)          

The dataset is shuffled and divided into three datasets:
1. train_data.csv for training
2. val_data.csv for validation
3. test_data.csv for testing

Import pandas library

In [28]:
import pandas as pd

Read datasets.

In [29]:
label_column = "Attack Type"

train = pd.read_csv("train_data.csv")
train_x = train.drop(label_column, axis=1)
train_y = train[label_column]

validation = pd.read_csv("val_data.csv")
validation_x = validation.drop(label_column, axis=1)
validation_y = validation[label_column]

test = pd.read_csv("test_data.csv")
test_x = test.drop(label_column, axis=1)
test_y = test[label_column]

Scale features, then train and tune kNN using the validation set. Fit the kNN model. Use validation data to fine tune the model. Compare scalers and kNN parameter sets, then keep the best validation result.

In [30]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from time import perf_counter

scalers = {
    "standard": StandardScaler(),
    "minmax": MinMaxScaler(),
}

param_grid = [
    {"n_neighbors": 3, "weights": "uniform", "metric": "euclidean"},
    {"n_neighbors": 5, "weights": "uniform", "metric": "euclidean"},
    {"n_neighbors": 5, "weights": "distance", "metric": "euclidean"},
    {"n_neighbors": 7, "weights": "distance", "metric": "euclidean"},
    {"n_neighbors": 9, "weights": "distance", "metric": "manhattan"},
]

results = []
best_result = None

for scaler_name, scaler in scalers.items():
    scaled_train_x = scaler.fit_transform(train_x)
    scaled_validation_x = scaler.transform(validation_x)

    for params in param_grid:
        # Fitting
        fit_start = perf_counter()
        knn = KNeighborsClassifier(**params)
        knn.fit(scaled_train_x, train_y)
        fit_time = perf_counter() - fit_start

        # Validation 
        predict_start = perf_counter()
        y_pred = knn.predict(scaled_validation_x)
        predict_time = perf_counter() - predict_start
        acc = accuracy_score(validation_y, y_pred)

        result = {
            "scaler_name": scaler_name,
            "scaler": scaler,
            "params": params,
            "accuracy": acc,
            "fit_time_sec": fit_time,
            "predict_time_sec": predict_time,
            "total_time_sec": fit_time + predict_time,
            "confusion_matrix": confusion_matrix(validation_y, y_pred),
            "report": classification_report(validation_y, y_pred),
            "model": knn,
        }
        results.append(result)

        if best_result is None or acc > best_result["accuracy"]:
            best_result = result

        print(f"\nScaler: {scaler_name}, params: {params}")
        print("accuracy:", acc)
        print("fit time (sec):", fit_time)
        print("predict time (sec):", predict_time)
        print(result["confusion_matrix"])
        print(result["report"])


Scaler: standard, params: {'n_neighbors': 3, 'weights': 'uniform', 'metric': 'euclidean'}
accuracy: 0.99295
fit time (sec): 0.12963884300006612
predict time (sec): 0.5352210430000923
[[    9     0     0     0     8     0     0]
 [    0    68     0     0     8     0     0]
 [    0     0  1013     7     3     0     0]
 [    0     0     7  1506    21     0     0]
 [    7     3    10    22 16542    31     1]
 [    0     0     0     1     5   709     0]
 [    0     0     0     0     7     0    12]]
                precision    recall  f1-score   support

          Bots       0.56      0.53      0.55        17
   Brute Force       0.96      0.89      0.93        76
          DDoS       0.98      0.99      0.99      1023
           DoS       0.98      0.98      0.98      1534
Normal Traffic       1.00      1.00      1.00     16616
 Port Scanning       0.96      0.99      0.97       715
   Web Attacks       0.92      0.63      0.75        19

      accuracy                           0.99     

Select best combination of parameters and scaler.

In [31]:
best_combination = max(range(len(results)), key=lambda i: results[i]["accuracy"])
best_result = results[best_combination]
best_scaler = best_result["scaler"]
best_params = best_result["params"]
scaled_train_x = best_scaler.fit_transform(train_x)
scaled_validation_x = best_scaler.transform(validation_x)
best_knn = KNeighborsClassifier(**best_params)
best_knn.fit(scaled_train_x, train_y)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",9
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'distance'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'manhattan'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [32]:
import pandas as pd
from IPython.display import display

results_table = pd.DataFrame([
    {
        "scaler": item["scaler_name"],
        "n_neighbors": item["params"]["n_neighbors"],
        "weights": item["params"]["weights"],
        "metric": item["params"]["metric"],
        "accuracy": item["accuracy"],
        "fit_time_sec": item["fit_time_sec"],
        "predict_time_sec": item["predict_time_sec"],
        "total_time_sec": item["total_time_sec"],
    }
    for item in results
])
results_table = results_table.sort_values(["accuracy", "total_time_sec"], ascending=[False, True])

print("All results ordered by accuracy and total time:")
display(results_table)
print("Best configuration:")
print(best_result["scaler_name"], best_result["params"])

All results ordered by accuracy and total time:


,scaler,n_neighbors,weights,metric,accuracy,fit_time_sec,predict_time_sec,total_time_sec
4,standard,9,distance,manhattan,0.99380,0.124735,0.862022,0.986757
2,standard,5,distance,euclidean,0.99330,0.124008,0.576370,0.700377
0,standard,3,uniform,euclidean,0.99295,0.129639,0.535221,0.664860
3,standard,7,distance,euclidean,0.99255,0.113785,0.608155,0.721940
9,minmax,9,distance,manhattan,0.99250,0.117574,0.482030,0.599604
7,minmax,5,distance,euclidean,0.99245,0.117735,0.343679,0.461414
8,minmax,7,distance,euclidean,0.99210,0.114485,0.378209,0.492695
5,minmax,3,uniform,euclidean,0.99175,0.153149,0.477726,0.630875
1,standard,5,uniform,euclidean,0.99150,0.133064,0.615474,0.748538
6,minmax,5,uniform,euclidean,0.98980,0.117274,0.343277,0.460551


Best configuration:
standard {'n_neighbors': 9, 'weights': 'distance', 'metric': 'manhattan'}


Evaluate the best kNN model on the test set.

In [33]:
scaled_test_x = best_scaler.transform(test_x)
y_pred = best_result["model"].predict(scaled_test_x)
print("Confusion Matrix:")
print(confusion_matrix(test_y, y_pred))
print("\nClassification Report:")
print(classification_report(test_y, y_pred))

Confusion Matrix:
[[    4     0     0     0    14     0     0]
 [    0    70     0     1     4     0     0]
 [    0     0  1016     5     2     0     0]
 [    0     1     1  1514    19     0     0]
 [    7     2     8    26 16533    40     0]
 [    0     0     0     2     2   711     0]
 [    0     0     0     0    12     0     6]]

Classification Report:
                precision    recall  f1-score   support

          Bots       0.36      0.22      0.28        18
   Brute Force       0.96      0.93      0.95        75
          DDoS       0.99      0.99      0.99      1023
           DoS       0.98      0.99      0.98      1535
Normal Traffic       1.00      1.00      1.00     16616
 Port Scanning       0.95      0.99      0.97       715
   Web Attacks       1.00      0.33      0.50        18

      accuracy                           0.99     20000
     macro avg       0.89      0.78      0.81     20000
  weighted avg       0.99      0.99      0.99     20000

